## Extração de Features — Overview

Calcula os ratios de desempenho por jogador por partida a partir do `charting-m-stats-Overview.csv`.
Ao final, salva `features_overview.csv` com as médias acumuladas pré-partida de cada jogador.

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

BASE = '../../new-dataset/tennis_MatchChartingProject-master'

overview = pd.read_csv(f'{BASE}/charting-m-stats-Overview.csv')
print('overview:', overview.shape)

overview: (56850, 20)


#### Filtrando apenas totais da partida (set == 'Total')

Preciso filtrar apenas o set total, porque é o resumo da partida

In [7]:
ov = overview.query("set == 'Total'").copy()
print(ov.shape)
ov.head(4)

(15116, 20)


,match_id,player,set,serve_pts,aces,dfs,first_in,first_won,second_in,second_won,bk_pts,bp_saved,return_pts,return_pts_won,winners,winners_fh,winners_bh,unforced,unforced_fh,unforced_bh
0,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,Jesper De Jong,Total,80,6,4,48,33,32,13,11,8,61,18,25,13,5,25,15,6
1,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,Michael Zheng,Total,61,3,2,34,27,27,16,3,3,80,34,28,23,2,21,11,8
7,20260517-M-Rome_Masters-F-Casper_Ruud-Jannik_S...,Casper Ruud,Total,64,3,2,37,24,27,13,5,2,55,16,22,13,6,22,11,9
8,20260517-M-Rome_Masters-F-Casper_Ruud-Jannik_S...,Jannik Sinner,Total,55,2,0,35,29,20,10,2,1,64,27,27,15,10,18,9,9


#### Calculando ratios de desempenho

Transformei os valores de algumas colunas para porcentagem. É melhor porque agrega algumas features e deixa escalado.

In [8]:
ov['first_serve_pct']      = ov['first_in']       / ov['serve_pts']
ov['first_serve_won_pct']  = ov['first_won']       / ov['first_in'].replace(0, np.nan)
ov['second_serve_won_pct'] = ov['second_won']      / ov['second_in'].replace(0, np.nan)
ov['ace_pct']              = ov['aces']            / ov['serve_pts']
ov['df_pct']               = ov['dfs']             / ov['serve_pts']
ov['return_won_pct']       = ov['return_pts_won']  / ov['return_pts'].replace(0, np.nan)
ov['winners_per_pt']       = ov['winners']         / (ov['serve_pts'] + ov['return_pts'])
ov['ue_per_pt']            = ov['unforced']        / (ov['serve_pts'] + ov['return_pts'])
ov['winners_fh_ratio']     = ov['winners_fh']      / ov['winners'].replace(0, np.nan)
ov['bp_save_pct']          = ov['bp_saved']        / ov['bk_pts'].replace(0, np.nan)

In [9]:
ratio_cols = ['match_id', 'player', 'first_serve_pct', 'first_serve_won_pct',
              'second_serve_won_pct', 'ace_pct', 'df_pct', 'return_won_pct',
              'winners_per_pt', 'ue_per_pt', 'winners_fh_ratio', 'bp_save_pct']

ov_ratios = ov[ratio_cols].copy()
print(ov_ratios.shape)
ov_ratios.head(4)

(15116, 12)


,match_id,player,first_serve_pct,first_serve_won_pct,second_serve_won_pct,ace_pct,df_pct,return_won_pct,winners_per_pt,ue_per_pt,winners_fh_ratio,bp_save_pct
0,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,Jesper De Jong,0.600000,0.687500,0.406250,0.075000,0.050000,0.295082,0.177305,0.177305,0.520000,0.727273
1,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,Michael Zheng,0.557377,0.794118,0.592593,0.049180,0.032787,0.425000,0.198582,0.148936,0.821429,1.000000
7,20260517-M-Rome_Masters-F-Casper_Ruud-Jannik_S...,Casper Ruud,0.578125,0.648649,0.481481,0.046875,0.031250,0.290909,0.184874,0.184874,0.590909,0.400000
8,20260517-M-Rome_Masters-F-Casper_Ruud-Jannik_S...,Jannik Sinner,0.636364,0.828571,0.500000,0.036364,0.000000,0.421875,0.226891,0.151261,0.555556,0.500000


Adicionando data

In [10]:
ov_ratios['Date'] = pd.to_datetime(
    ov_ratios['match_id'].str.split('-').str[0], format='%Y%m%d', errors='coerce'
)

print(ov_ratios.shape)
ov_ratios.head(3)

(15116, 13)


,match_id,player,first_serve_pct,first_serve_won_pct,second_serve_won_pct,ace_pct,df_pct,return_won_pct,winners_per_pt,ue_per_pt,winners_fh_ratio,bp_save_pct,Date
0,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,Jesper De Jong,0.600000,0.687500,0.406250,0.075000,0.050000,0.295082,0.177305,0.177305,0.520000,0.727273,2026-05-21
1,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,Michael Zheng,0.557377,0.794118,0.592593,0.049180,0.032787,0.425000,0.198582,0.148936,0.821429,1.000000,2026-05-21
7,20260517-M-Rome_Masters-F-Casper_Ruud-Jannik_S...,Casper Ruud,0.578125,0.648649,0.481481,0.046875,0.031250,0.290909,0.184874,0.184874,0.590909,0.400000,2026-05-17


Organizando as médias dos jogadores e fazendo a subir de acordo com os seus jogos

In [11]:
ov_ratios = ov_ratios.sort_values(['player', 'Date'])

ratio_feature_cols = ['first_serve_pct', 'first_serve_won_pct', 'second_serve_won_pct',
                      'ace_pct', 'df_pct', 'return_won_pct', 'winners_per_pt',
                      'ue_per_pt', 'winners_fh_ratio', 'bp_save_pct']

for col in ratio_feature_cols:
    ov_ratios[f'avg_{col}'] = (
        ov_ratios.groupby('player')[col].transform(lambda x: x.expanding().mean().shift(1))
    )

print(ov_ratios.shape)
ov_ratios.head(4)

(15116, 23)


,match_id,player,first_serve_pct,first_serve_won_pct,second_serve_won_pct,ace_pct,df_pct,return_won_pct,winners_per_pt,ue_per_pt,...,avg_first_serve_pct,avg_first_serve_won_pct,avg_second_serve_won_pct,avg_ace_pct,avg_df_pct,avg_return_won_pct,avg_winners_per_pt,avg_ue_per_pt,avg_winners_fh_ratio,avg_bp_save_pct
54719,19890909-M-US_Open-SF-Boris_Becker-Aaron_Krick...,Aaron Krickstein,0.447917,0.720930,0.377358,0.062500,0.041667,0.414414,0.149758,0.149758,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
54522,19900415-M-Tokyo_Outdoor-F-Aaron_Krickstein-St...,Aaron Krickstein,0.705128,0.618182,0.347826,0.038462,0.000000,0.388889,0.153333,0.133333,...,0.447917,0.720930,0.377358,0.062500,0.041667,0.414414,0.149758,0.149758,0.451613,0.400000
53540,19910902-M-US_Open-R16-Aaron_Krickstein-Jimmy_...,Aaron Krickstein,0.557214,0.660714,0.573034,0.034826,0.009950,0.385965,0.096774,0.139785,...,0.576522,0.669556,0.362592,0.050481,0.020833,0.401652,0.151546,0.141546,0.399719,0.543750
53438,19911026-M-Stockholm_Masters-SF-Aaron_Krickste...,Aaron Krickstein,0.571429,0.535714,0.380952,0.081633,0.000000,0.266667,0.095745,0.117021,...,0.570086,0.666609,0.432739,0.045262,0.017206,0.396423,0.133289,0.140959,0.396109,0.626389


#### Selecionando apenas as colunas finais (avg_*)

In [12]:
avg_cols = [f'avg_{c}' for c in ratio_feature_cols]
features_overview = ov_ratios[['match_id', 'player', 'Date'] + avg_cols].copy()

print(features_overview.shape)
print(f'NaN na primeira partida de cada jogador (esperado): {features_overview["avg_first_serve_pct"].isna().sum()}')
features_overview.head(4)

(15116, 13)
NaN na primeira partida de cada jogador (esperado): 1002


,match_id,player,Date,avg_first_serve_pct,avg_first_serve_won_pct,avg_second_serve_won_pct,avg_ace_pct,avg_df_pct,avg_return_won_pct,avg_winners_per_pt,avg_ue_per_pt,avg_winners_fh_ratio,avg_bp_save_pct
54719,19890909-M-US_Open-SF-Boris_Becker-Aaron_Krick...,Aaron Krickstein,1989-09-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
54522,19900415-M-Tokyo_Outdoor-F-Aaron_Krickstein-St...,Aaron Krickstein,1990-04-15,0.447917,0.720930,0.377358,0.062500,0.041667,0.414414,0.149758,0.149758,0.451613,0.400000
53540,19910902-M-US_Open-R16-Aaron_Krickstein-Jimmy_...,Aaron Krickstein,1991-09-02,0.576522,0.669556,0.362592,0.050481,0.020833,0.401652,0.151546,0.141546,0.399719,0.543750
53438,19911026-M-Stockholm_Masters-SF-Aaron_Krickste...,Aaron Krickstein,1991-10-26,0.570086,0.666609,0.432739,0.045262,0.017206,0.396423,0.133289,0.140959,0.396109,0.626389


#### Salvando

In [13]:
features_overview.to_csv(f'{BASE}/features_overview.csv', index=False)
print('Salvo:', f'{BASE}/features_overview.csv')
print('Shape:', features_overview.shape)

Salvo: ../../new-dataset/tennis_MatchChartingProject-master/features_overview.csv
Shape: (15116, 13)


##### Criando o dataset dos vencedores de cada partida

In [14]:
matches = pd.read_csv(f'{BASE}/charting-m-matches.csv')
matches.shape

(7566, 15)

In [24]:
matches['Final TB?'].value_counts()

Final TB?
1     5719
0     1003
A      695
T       59
N       20
S       14
V       10
A        3
W        1
Name: count, dtype: int64

In [ ]:
#Basicamente, preciso dar um jeito de agrupar por jogo, set e jogador. Em seguida, 

,match_id,player,set,serve_pts,aces,dfs,first_in,first_won,second_in,second_won,bk_pts,bp_saved,return_pts,return_pts_won,winners,winners_fh,winners_bh,unforced,unforced_fh,unforced_bh
0,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,Jesper De Jong,Total,80,6,4,48,33,32,13,11,8,61,18,25,13,5,25,15,6
1,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,Michael Zheng,Total,61,3,2,34,27,27,16,3,3,80,34,28,23,2,21,11,8
2,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,Jesper De Jong,1,52,6,2,32,22,20,8,7,6,34,9,18,8,3,13,7,4
3,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,Michael Zheng,1,34,2,1,19,14,15,11,0,0,52,22,18,15,1,12,8,3
4,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,Jesper De Jong,2,28,0,2,16,11,12,5,4,2,27,9,7,5,2,12,8,2


In [ ]:
ov_total = overview.query("set == 'Total'").copy()
ov_total = ov_total.merge(
    matches[['match_id', 'Player 1', 'Player 2']],
    on='match_id', how='left'
)

p1 = ov_total[ov_total['player'] == ov_total['Player 1']][['match_id', 'serve_pts', 'return_pts_won']].rename(
    columns={'serve_pts': 'p1_serve_pts', 'return_pts_won': 'p1_ret_won'})
p2 = ov_total[ov_total['player'] == ov_total['Player 2']][['match_id', 'serve_pts', 'return_pts_won']].rename(
    columns={'serve_pts': 'p2_serve_pts', 'return_pts_won': 'p2_ret_won'})

winners = p1.merge(p2, on='match_id')
winners['p1_total_pts'] = (winners['p1_serve_pts'] - winners['p2_ret_won']) + winners['p1_ret_won']
winners['p2_total_pts'] = (winners['p2_serve_pts'] - winners['p1_ret_won']) + winners['p2_ret_won']
winners['p1_wins'] = (winners['p1_total_pts'] > winners['p2_total_pts']).astype(int)

# descartar empates de pontos (~87 casos)
winners = winners[winners['p1_total_pts'] != winners['p2_total_pts']][['match_id', 'p1_wins']]

print(winners.shape)
print(winners['p1_wins'].value_counts())